<a href="https://colab.research.google.com/github/23b2232/AI-guard-room-system/blob/main/agent2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **AI GUARD ROOM AGENT**

https://drive.google.com/drive/folders/15ianZxhwL1nsAzN3ZzCvwubnBTOtSSv7?usp=sharing

**Srishti Gupta 23b2520** \
**Yashasvee Taiwade 23b2232**

(WITH PASSCODE & ESCALATING DIALOGUE)

This agent provides three-tier security:

Face Recognition - Identifies pre-enrolled authorized users \
Passcode Authentication - Allows room owner to grant temporary access via secret phrase \
Escalating Dialogue - Professional interaction that intensifies based on visitor behavior

Features:
- 3-level escalating dialogue (polite → firm → ultimatum)
- Detects forcing entry attempts
- Grants access with correct passcode

**System Flow**

1. Video Processing \
   ↓
2. Face Detection (every 30 frames) \
   ↓
3. Face Recognition \
   ↓
4. a. AUTHORIZED → Welcome Message → Grant Access \
   ↓
4. b. UNAUTHORIZED → Escalation Protocol \
   ↓
5. Level 1: Polite Inquiry \
   ├── Passcode Correct? → Grant Access \
   ├── Force Entry? → Jump to Level 3 \
   └── Neither → Continue to Level 2 \
   ↓
6. Level 2: Firm Request \
   ├── Passcode Correct? → Grant Access \
   ├── Force Entry? → Jump to Level 3 \
   └── Neither → Continue to Level 3 \
   ↓
7. Level 3: Final Warning → Security Alert

bash commands to install required dependencies

pip install opencv-python numpy gtts pygame deepface SpeechRecognition pyaudio tensorflow tf-keras \
brew install portaudio

In [ ]:
import cv2
import time
import numpy as np
import os
from gtts import gTTS
import pygame
import speech_recognition as sr
from deepface import DeepFace
import re

/opt/anaconda3/envs/ai_agent/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.10.18)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
known_faces_path = "known_faces"

CONFIGURATION - secret passcode set here

If an unauthorized person says the passcode, they're granted access \
Works even if they say extra words, for example: "Hi, the passcode is alpha"

In [ ]:
SECRET_PASSCODE = "alpha"
# The user must say this phrase (not case-sensitive)

In [ ]:
def speak(text):
    """Converts text to speech and plays it."""
    try:
        import pygame
        tts = gTTS(text=text, lang='en', tld='com', slow=False)
        audio_file = "response.mp3"
        tts.save(audio_file)

        pygame.mixer.init()
        pygame.mixer.music.load(audio_file)
        pygame.mixer.music.play()

        while pygame.mixer.music.get_busy():
            time.sleep(0.1)

        pygame.mixer.quit()
        os.remove(audio_file)
    except Exception as e:
        print(f"Speech error: {e}")
        print(f"Agent says: {text}")

**enroll_trusted_users**

Purpose: Loads and processes facial embeddings for authorized personnel.

Process:

Scans directories in known_faces/ \
Generates 2048-dimensional embeddings using VGG-Face model \
Stores one embedding per person (first valid image found)

In [ ]:
def enroll_trusted_users(known_faces_path='known_faces'):
    """Enrolls trusted users by their face images."""
    known_embeddings = {}

    if not os.path.exists(known_faces_path):
        print(f"Error: Directory '{known_faces_path}' not found.")
        return known_embeddings

    print(f"Enrolling trusted users from '{known_faces_path}' directory...")

    for name in os.listdir(known_faces_path):
        person_dir = os.path.join(known_faces_path, name)
        if os.path.isdir(person_dir):
            for filename in os.listdir(person_dir):
                if filename.lower().endswith((".jpg", ".png", ".jpeg")):
                    img_path = os.path.join(person_dir, filename)
                    try:
                        result = DeepFace.represent(
                            img_path=img_path,
                            model_name="VGG-Face",
                            enforce_detection=False
                        )
                        embedding = result[0]['embedding']
                        known_embeddings[name] = embedding
                        print(f"✓ Enrolled: {name}")
                        break
                    except Exception as e:
                        print(f"✗ Failed to enroll {name}: {e}")

    print(f"Total enrolled users: {len(known_embeddings)}\n")
    return known_embeddings

**identify_user**

Purpose: Compares detected face against enrolled users.

Algorithm:

Calculates Euclidean distance between embeddings \
Returns closest match if distance < threshold \
Returns "Unauthorised" if no match found

Key Insight: Lower distance = higher similarity. Threshold of 0.6 typically distinguishes between same person vs. different people.

Recognition Threshold

Lower (0.4-0.5): More lenient, may allow similar faces \
Higher (0.7-0.8): More strict, may reject authorized users \
Default (0.6): Balanced for most use cases

In [ ]:
def identify_user(face_embedding, known_embeddings, threshold=0.6):
    """Identifies a user based on their face embedding."""
    min_distance = float('inf')
    identified_name = "Unauthorised"

    for name, known_embedding in known_embeddings.items():
        distance = np.linalg.norm(np.array(face_embedding) - np.array(known_embedding))
        if distance < min_distance:
            min_distance = distance
            if distance < threshold:
                identified_name = name

    print(f"  Distance: {min_distance:.4f} (threshold: {threshold})")
    return identified_name

**listen_for_response**

Purpose: Captures and transcribes user's spoken input.

Configuration:

energy_threshold = 3000 - Sensitivity to detect speech vs. noise \
pause_threshold = 1.5 - Seconds of silence before considering speech complete \
phrase_time_limit = 15 - Maximum recording duration per response

In [ ]:
def listen_for_response(timeout=15):
    """Listens for the user's voice and returns the transcribed text."""
    r = sr.Recognizer()
    r.energy_threshold = 3000
    r.dynamic_energy_threshold = True
    r.pause_threshold = 1.5

    try:
        with sr.Microphone() as source:
            print("\nListening... (you have 15 seconds)")
            r.adjust_for_ambient_noise(source, duration=1)
            audio = r.listen(source, timeout=timeout, phrase_time_limit=15)

        print("Processing speech...")
        response = r.recognize_google(audio, language='en-US')
        print(f"You said: '{response}'\n")
        return response

    except sr.UnknownValueError:
        print("Could not understand audio\n")
        return ""
    except sr.WaitTimeoutError:
        print("No speech detected\n")
        return ""
    except Exception as e:
        if "Bad CPU" not in str(e):
            print(f"Error: {e}\n")
        return ""

**check_passcode**

Purpose: Validates if user's spoken response contains the secret passcode.

Features:

Normalizes whitespace and case for flexible matching \
Allows passcode within natural speech (e.g., "my code is alpha") \
Returns boolean for clean conditional logic

In [ ]:
def check_passcode(user_input, secret_passcode):
    """Check if user input contains the secret passcode."""
    if not user_input:
        return False

    # Normalize both strings (lowercase, remove extra spaces)
    user_normalized = re.sub(r'\s+', ' ', user_input.lower().strip())
    passcode_normalized = secret_passcode.lower().strip()

    # Check if passcode is in the user's response
    return passcode_normalized in user_normalized

**detect_force_entry_attempt**

Purpose: Identifies aggressive or non-compliant behavior through keyword analysis.

Trigger Phrases:

Direct commands: "let me in", "open the door", "move" \
Dismissive language: "don't care", "whatever" \
Physical assertions: "coming through", "step aside"

Result: Immediately escalates to Level 3 (ultimatum) when detected.

In [ ]:
def detect_force_entry_attempt(user_input):
    """Detect if user is trying to force entry."""
    if not user_input:
        return False

    force_keywords = [
        "let me in", "open the door", "move", "get out of my way",
        "i'm going in", "step aside", "coming through", "coming in",
        "don't care", "whatever", "who cares"
    ]

    user_lower = user_input.lower()
    return any(keyword in user_lower for keyword in force_keywords)

**handle_unauthorized_dialogue**

Purpose: Manages the three-level escalation protocol for unauthorized visitors.

Escalation Levels

Level 1: Professional Inquiry \
Tone: Polite but firm \
Goal: Gather information (name, purpose) \
Listens for: Passcode or cooperative response

Level 2: Authorization Request \
Tone: Direct and assertive \
Goal: Explicitly request credentials \
Listens for: Passcode or continued non-compliance

Level 3: Final Warning \
Tone: Authoritative ultimatum \
Goal: Deter entry, notify of consequences \
Action: Logs security alert, no further dialogue

Special Case: Force Entry Detection \
If aggressive behavior detected at Level 1 or 2 → immediate jump to Level 3

Return Values: \
"PASSCODE_GRANTED" - Access allowed via correct passcode \
"SECURITY_CALLED" - Access denied, security notified

In [ ]:
def handle_unauthorized_dialogue(secret_passcode):
    """Handle escalating dialogue with unauthorized person."""

    # Escalation Level 1: Polite but professional
    dialogue_l1 = {
        "prompt": "Hello. I don't recognize you as an authorized person. Please state your name and the purpose of your visit.",
        "wait_for_response": True
    }

    # Escalation Level 2: Firm and direct
    dialogue_l2 = {
        "prompt": "I need proper authorization to grant you access. Do you have an access code?",
        "wait_for_response": True
    }

    # Escalation Level 3: Final warning / Ultimatum
    dialogue_l3 = {
        "prompt": "This is your final warning. This is a restricted area. You do not have authorization. Security has been alerted and will arrive shortly. Do not attempt to proceed.",
        "wait_for_response": False
    }

    dialogue_sequence = [dialogue_l1, dialogue_l2, dialogue_l3]

    print("UNAUTHORIZED PERSON DETECTED")
    print("\nInitiating security protocol...\n")

    for level, dialogue in enumerate(dialogue_sequence, 1):
        print(f"ESCALATION LEVEL {level}/3")
        print(f"\nAgent: {dialogue['prompt']}")
        speak(dialogue['prompt'])

        if dialogue['wait_for_response']:
            user_response = listen_for_response()

            # Check for passcode
            if check_passcode(user_response, secret_passcode):
                print("PASSCODE VERIFIED")
                success_msg = "Access code verified. Welcome. You may proceed."
                print(f"\nAgent: {success_msg}\n")
                speak(success_msg)
                return "PASSCODE_GRANTED"

            # Check for force entry attempt
            if detect_force_entry_attempt(user_response):
                print("\nFORCING ENTRY DETECTED - SKIPPING TO FINAL WARNING\n")
                # Skip to level 3
                print("ESCALATION LEVEL 3/3 - FINAL WARNING")
                print(f"\nAgent: {dialogue_l3['prompt']}")
                speak(dialogue_l3['prompt'])
                time.sleep(2)
                return "SECURITY_CALLED"

            # If no passcode and not forcing entry, continue to next level
            if level < 3:
                print(f"Escalating to Level {level + 1}...")
                time.sleep(1)
        else:
            # Level 3 - just wait a moment
            time.sleep(2)

    print("SECURITY ALERT: Unauthorized access attempt logged")
    return "SECURITY_CALLED"

Scenario A - Correct Passcode:

Agent: "I don't recognize you. State your name and purpose." \
You: "I'm here for a meeting, passcode is alpha" \
Agent: "Access code verified. Welcome. You may proceed."

\

Scenario B - Cooperative but no passcode:

Level 1: "State your name and purpose" \
You: "I'm John, here to visit" \
Level 2: "Do you have an access code?" \
You: "No, I don't have one" \
Level 3: "Final warning. Security alerted."

\

Scenario C - Forcing Entry:

Agent: "State your name and purpose" \
You: "I don't care, let me in" \
Agent: [SKIPS TO LEVEL 3] "Final warning. Do not proceed."

**Frame Sampling Rationale**

check_interval = 30  # Process every 30th frame


At 60 FPS, checks faces twice per second \
Reduces computational load by ~97% \
Maintains real-time responsiveness


**Image Resizing Rationale**

pythonsmall_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)


Reduces pixel count by 75% \
Faster face detection and embedding generation \
Minimal impact on recognition accuracy

In [ ]:
def main():
    """Main loop for the video-based AI Guard Room system."""

    print("AI GUARD ROOM AGENT v2.0")
    print()

    # Enroll trusted users
    known_embeddings = enroll_trusted_users(known_faces_path)

    if not known_embeddings:
        print("Warning: No trusted users enrolled. All faces will be unauthorized.\n")

    video_path = "entry_video.mp4"

    if not os.path.exists(video_path):
        print(f"Error: Video file '{video_path}' not found\n")
        return

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Error: Could not open video file\n")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video loaded: {total_frames} frames @ {fps:.1f} FPS\n")

    interaction_complete = False
    frame_count = 0
    check_interval = 30

    print("Starting video analysis...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("\nEnd of video reached.")
            break

        frame_count += 1

        if frame_count % check_interval != 0:
            continue

        progress = (frame_count / total_frames) * 100
        print(f"Processing frame {frame_count}/{total_frames} ({progress:.1f}%)", end="\r")

        try:
            small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)

            faces = DeepFace.extract_faces(
                img_path=small_frame,
                detector_backend='opencv',
                enforce_detection=False
            )

            if faces and len(faces) > 0 and not interaction_complete:
                print(f"\n\nFace detected at frame {frame_count}")

                face_data = faces[0]
                face_img = face_data['face']

                result = DeepFace.represent(
                    img_path=face_img,
                    model_name="VGG-Face",
                    enforce_detection=False
                )
                face_embedding = result[0]['embedding']

                identified_user = identify_user(face_embedding, known_embeddings)

                if identified_user == "Unauthorised":
                    # Start escalating dialogue with passcode checking
                    result = handle_unauthorized_dialogue(SECRET_PASSCODE)

                    if result == "PASSCODE_GRANTED":
                        print("Access granted via passcode\n")
                    else:
                        print("Access denied - Security notified\n")

                    interaction_complete = True
                    break

                else:
                    # Authorized user
                    print(f"\nAUTHORIZED USER: {identified_user}")
                    welcome_msg = f"Hello {identified_user}. Welcome back. Access granted."
                    print(f"Agent: {welcome_msg}\n")
                    speak(welcome_msg)
                    interaction_complete = True
                    time.sleep(2)
                    break

        except Exception as e:
            print(f"\nProcessing error at frame {frame_count}: {e}")

        if interaction_complete:
            break

    cap.release()
    print("System Shutdown")

if __name__ == "__main__":
    main()

AI GUARD ROOM AGENT v2.0

Enrolling trusted users from 'known_faces' directory...
✓ Enrolled: a
✓ Enrolled: b
Total enrolled users: 2

Video loaded: 881 frames @ 60.1 FPS

Starting video analysis...
Processing frame 30/881 (3.4%)

Face detected at frame 30
  Distance: 1.3500 (threshold: 0.6)
UNAUTHORIZED PERSON DETECTED

Initiating security protocol...

ESCALATION LEVEL 1/3

Agent: Hello. I don't recognize you as an authorized person. Please state your name and the purpose of your visit.

Listening... (you have 15 seconds)
Processing speech...
You said: 'let me in I don't care'


FORCING ENTRY DETECTED - SKIPPING TO FINAL WARNING

ESCALATION LEVEL 3/3 - FINAL WARNING

Agent: This is your final warning. This is a restricted area. You do not have authorization. Security has been alerted and will arrive shortly. Do not attempt to proceed.
Access denied - Security notified

System Shutdown
